# Quick setup via docker

In [ ]:
!docker run -d \
  --name my-mongo \
  -p 27017:27017 \
  -v mongo_data:/data/db \
  mongo:latest

Explanation:
- -p 27017:27017 → exposes MongoDB on your local machine
- -v mongo_data:/data/db → stores data persistently
- mongo:7 → current stable MongoDB image

In [ ]:
!docker ps

In [ ]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017")

db = client["testdb"]            # Create/open a database
collection = db["users"] 

In [ ]:
# Insert a document
collection.insert_one({"name": "Alice", "age": 30})

In [ ]:
# Fetch documents
for user in collection.find():
    print(user)

lets delete the container for a cleaner solution

In [ ]:
!docker rm -f my-mongo

# Run MongoDB via docker-compose

In [7]:
%%writefile docker-compose.yaml
services:
  mongo:
    image: mongo:latest
    restart: always
    ports:
      - "27017:27017"
    volumes:
      - mongo_data:/data/db

volumes:
  mongo_data:

Overwriting docker-compose.yaml


In [8]:
!docker-compose up -d

[+] Running 2/3
 ✔ Network 17text-to-queryagentswithmongodbandlanggraph_default    Created 0.0s 
 ✔ Volume 17text-to-queryagentswithmongodbandlanggraph_mongo_data  Created 0.0s 
 ⠋ Container 17text-to-queryagentswithmongodbandlanggraph-mongo-1  Creating0.0s 
[+] Running 2/3
 ✔ Network 17text-to-queryagentswithmongodbandlanggraph_default    Created 0.0s 
 ✔ Volume 17text-to-queryagentswithmongodbandlanggraph_mongo_data  Created 0.0s 
 ⠙ Container 17text-to-queryagentswithmongodbandlanggraph-mongo-1  Starting0.1s 
[+] Running 2/3
 ✔ Network 17text-to-queryagentswithmongodbandlanggraph_default    Created 0.0s 
 ✔ Volume 17text-to-queryagentswithmongodbandlanggraph_mongo_data  Created 0.0s 
 ⠹ Container 17text-to-queryagentswithmongodbandlanggraph-mongo-1  Starting0.2s 
[+] Running 2/3
 ✔ Network 17text-to-queryagentswithmongodbandlanggraph_default    Created 0.0s 
 ✔ Volume 17text-to-queryagentswithmongodbandlanggraph_mongo_data  Created 0.0s 
 ⠸ Container 17text-to-queryagentswithmongodb

This creates persistent storage using a volume `mongo_data`

In [9]:
!docker volume ls

DRIVER    VOLUME NAME
local     17text-to-queryagentswithmongodbandlanggraph_mongo_data
local     902a055de83e020732d46e374b9ce679d2c4830cc709a9698364fd2d2947ca1a
local     data
local     deployment_langgraph-data
local     minikube
local     mongo_data
local     mysql-project_db_data
local     myvol103


In [10]:
!docker-compose ps

NAME                                                   IMAGE          COMMAND                  SERVICE   CREATED         STATUS         PORTS
17text-to-queryagentswithmongodbandlanggraph-mongo-1   mongo:latest   "docker-entrypoint.s…"   mongo     3 seconds ago   Up 2 seconds   0.0.0.0:27017->27017/tcp, [::]:27017->27017/tcp


`docker-compose down -v` will remove any stopped containers and also remove the volume (leave it out if you want to persist the data)

In [11]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017")

db = client["testdb"]            # Create/open a database
collection = db["users"] 

# Insert a document
collection.insert_one({"name": "Alice", "age": 30})

# Fetch documents
for user in collection.find():
    print(user)

{'_id': ObjectId('693aebe6aba91d2e182abd73'), 'name': 'Alice', 'age': 30}


In [6]:
!docker-compose down -v

[+] Running 0/1
 ⠋ Container 17text-to-queryagentswithmongodbandlanggraph-mongo-1  Stopping0.1s 
[+] Running 0/1
 ⠙ Container 17text-to-queryagentswithmongodbandlanggraph-mongo-1  Stopping0.2s 
[+] Running 0/1
 ⠹ Container 17text-to-queryagentswithmongodbandlanggraph-mongo-1  Stopping0.3s 
[+] Running 0/1
 ⠸ Container 17text-to-queryagentswithmongodbandlanggraph-mongo-1  Stopping0.4s 
[+] Running 0/1
 ⠼ Container 17text-to-queryagentswithmongodbandlanggraph-mongo-1  Stopping0.5s 
[+] Running 0/1
 ⠴ Container 17text-to-queryagentswithmongodbandlanggraph-mongo-1  Stopping0.6s 
[+] Running 2/3
 ✔ Container 17text-to-queryagentswithmongodbandlanggraph-mongo-1  Removed 0.7s 
 ✔ Volume 17text-to-queryagentswithmongodbandlanggraph_mongo_data  Removed 0.0s 
 ⠋ Network 17text-to-queryagentswithmongodbandlanggraph_default    Removing0.0s 
[+] Running 2/3
 ✔ Container 17text-to-queryagentswithmongodbandlanggraph-mongo-1  Removed 0.7s 
 ✔ Volume 17text-to-queryagentswithmongodbandlanggraph_mongo_d

# Download some data sets

`curl -LO https://atlas-education.s3.amazonaws.com/sampledata.archive` to download the datasets

In [12]:
!docker-compose up -d

[+] Running 1/1
 ✔ Container 17text-to-queryagentswithmongodbandlanggraph-mongo-1  Running 0.0s 


In [14]:
!docker ps

CONTAINER ID   IMAGE          COMMAND                  CREATED          STATUS          PORTS                                             NAMES
ee526f0fafd7   mongo:latest   "docker-entrypoint.s…"   10 minutes ago   Up 10 minutes   0.0.0.0:27017->27017/tcp, [::]:27017->27017/tcp   17text-to-queryagentswithmongodbandlanggraph-mongo-1


copy the data on the container in the `tmp` directory for this you need the container name (see above)

In [15]:
!docker cp sampledata.archive 17text-to-queryagentswithmongodbandlanggraph-mongo-1:/tmp/sampledata.archive

Successfully copied 370MB to 17text-to-queryagentswithmongodbandlanggraph-mongo-1:/tmp/sampledata.archive


This uses mongorestore to import all the sample databases into your local MongoDB instance

In [17]:
!docker exec -it 17text-to-queryagentswithmongodbandlanggraph-mongo-1 mongorestore \
    --archive=/tmp/sampledata.archive

2025-12-11T16:19:22.364+0000	preparing collections to restore from
2025-12-11T16:19:22.375+0000	reading metadata for sample_mflix.users from archive '/tmp/sampledata.archive'
2025-12-11T16:19:22.375+0000	reading metadata for sample_weatherdata.data from archive '/tmp/sampledata.archive'
2025-12-11T16:19:22.375+0000	reading metadata for sample_analytics.transactions from archive '/tmp/sampledata.archive'
2025-12-11T16:19:22.375+0000	reading metadata for sample_training.routes from archive '/tmp/sampledata.archive'
2025-12-11T16:19:22.375+0000	reading metadata for sample_training.inspections from archive '/tmp/sampledata.archive'
2025-12-11T16:19:22.375+0000	reading metadata for sample_training.grades from archive '/tmp/sampledata.archive'
2025-12-11T16:19:22.375+0000	reading metadata for sample_geospatial.shipwrecks from archive '/tmp/sampledata.archive'
2025-12-11T16:19:22.375+0000	reading metadata for sample_mflix.embedded_movies from archive '/tmp/sampledata.archive'
2025-12-11T16:19

You can confirm the data was restore correctly by connecting and listing databases:

In [21]:
!docker exec -it 17text-to-queryagentswithmongodbandlanggraph-mongo-1 mongosh --eval "show dbs"

]0;mongosh mongodb://127.0.0.1:27017/?directConnection=true&serverSelectionTimeoutMS=2000admin                40.00 KiB
config              108.00 KiB
local                40.00 KiB
sample_airbnb        51.84 MiB
sample_analytics      9.00 MiB
sample_geospatial   980.00 KiB
sample_guides        40.00 KiB
sample_mflix         94.65 MiB
sample_restaurants    5.91 MiB
sample_supplies     968.00 KiB
sample_training      40.42 MiB
sample_weatherdata    2.38 MiB
testdb               40.00 KiB


In [22]:
from pymongo import MongoClient
from pprint import pprint

# ----------------------------
# 1️⃣ Connect to local MongoDB
# ----------------------------
client = MongoClient("mongodb://localhost:27017")

# Access the sample_mflix database
db = client["sample_mflix"]

# ----------------------------
# 2️⃣ List collections
# ----------------------------
print("Collections in sample_mflix:")
collections = db.list_collection_names()
for col in collections:
    print("-", col)

# ----------------------------
# 3️⃣ Fetch a random sample from movies
# ----------------------------
print("\nRandom sample of 5 movies:")
random_movies = db.movies.aggregate([{"$sample": {"size": 5}}])
for movie in random_movies:
    pprint(movie)

# ----------------------------
# 4️⃣ Optional: Fetch first 5 movies
# ----------------------------
print("\nFirst 5 movies:")
for movie in db.movies.find().limit(5):
    pprint(movie)


Collections in sample_mflix:
- users
- movies
- comments
- theaters
- embedded_movies
- sessions

Random sample of 5 movies:
{'_id': ObjectId('573a13b7f29313caabd498c5'),
 'awards': {'nominations': 3, 'text': '3 nominations.', 'wins': 0},
 'cast': ['Kunal Khemu', 'Deepal Shaw', 'Smiley Suri', 'Atul Parchure'],
 'countries': ['India'],
 'directors': ['Mohit Suri'],
 'fullplot': '18 years ago, the Darr family, consisting of Pushkaran and his '
             'son, Kunal, were forced to leave Kashmir by terrorists, who had '
             'forced thousands of other Kashmiri Pandits to be mere refugees '
             'in their very own country. Pushkaran and Kunal re-locate to '
             'Bombay, where they live in a small room. This is where Kunal '
             'grows up, & gets a job at a gym. Then one day, the police knock '
             'on his door, informing him that his father had lost his hold '
             'from a crowded local train, fallen, and instantly killed. A '
         